# PA2 - Machine Learning con Car Evaluation Dataset

**Nombre:** Anthony Baldoceda  
**Código ISIL:** Colocar aquí tu código ISIL

## Objetivo
Desarrollar un modelo de machine learning para predecir la evaluación de un auto usando el dataset Car Evaluation de UCI.

In [ ]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


## Parte 1: Dataset

Se utiliza el dataset **Car Evaluation** de UCI Machine Learning Repository. Este dataset contiene características categóricas de autos y permite predecir la evaluación final del vehículo.

El modelo aprenderá a predecir la clase de evaluación del auto:

- `unacc`: inaceptable
- `acc`: aceptable
- `good`: bueno
- `vgood`: muy bueno

Las variables predictoras son: `buying`, `maint`, `doors`, `persons`, `lug_boot` y `safety`.

In [ ]:
# Fetch dataset
car_evaluation = fetch_ucirepo(id=19)

# Data as pandas DataFrames
X = car_evaluation.data.features
y = car_evaluation.data.targets

# Unimos variables predictoras y objetivo en df
df = pd.concat([X, y], axis=1)

df.head()

In [ ]:
print(car_evaluation.metadata)
print(car_evaluation.variables)

## Análisis exploratorio de datos

In [ ]:
df.info()
print('Shape:', df.shape)
df.describe(include='all')

### Gráfica 1: Distribución de la variable objetivo
Esta gráfica permite conocer cuántos registros existen por cada clase. Es importante para identificar si el dataset está balanceado o si una clase predomina sobre las demás.

In [ ]:
target_col = y.columns[0]

plt.figure(figsize=(7,4))
sns.countplot(data=df, x=target_col)
plt.title('Distribución de la variable objetivo')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.show()

### Gráfica 2: Evaluación según seguridad
Esta gráfica permite analizar cómo cambia la evaluación del auto según el nivel de seguridad. Es relevante porque la seguridad suele ser una variable muy influyente en la clasificación final.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='safety', hue=target_col)
plt.title('Evaluación del auto según nivel de seguridad')
plt.xlabel('Seguridad')
plt.ylabel('Cantidad')
plt.show()

### Gráfica 3: Evaluación según precio de compra
Esta gráfica ayuda a observar si el precio de compra influye en la clasificación del auto. Permite comparar la distribución de clases para autos con precio bajo, medio, alto o muy alto.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='buying', hue=target_col)
plt.title('Evaluación del auto según precio de compra')
plt.xlabel('Precio de compra')
plt.ylabel('Cantidad')
plt.show()

## Procesamiento de datos

El dataset contiene variables categóricas, por lo que se aplica `OneHotEncoder` para convertirlas en variables numéricas. Este paso es necesario porque los modelos de machine learning de scikit-learn trabajan con valores numéricos.

Además, se verifica si existen valores nulos.

In [ ]:
print(df.isnull().sum())

X = car_evaluation.data.features
y = car_evaluation.data.targets.squeeze()

categorical_cols = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

## Parte 2: División de datos

Se utiliza `stratify=y` para conservar la proporción de clases en entrenamiento y prueba. Esto es importante porque la variable objetivo tiene varias clases y no todas tienen la misma frecuencia.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)

## Entrenamiento de modelos

Se entrenan dos modelos de clasificación:

1. Árbol de Decisión
2. Random Forest

In [ ]:
modelo_arbol = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=8, random_state=42))
])

modelo_random_forest = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

modelo_arbol.fit(X_train, y_train)
modelo_random_forest.fit(X_train, y_train)

## Evaluación de modelos

Se evalúan las métricas `accuracy`, `precision`, `recall` y `f1-score`. Como el problema tiene varias clases, se usa el promedio `macro` para comparar el rendimiento global entre clases.

In [ ]:
modelos = {
    'Árbol de Decisión': modelo_arbol,
    'Random Forest': modelo_random_forest
}

resultados = []

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    cv_score = cross_val_score(modelo, X, y, cv=5, scoring='accuracy').mean()
    
    resultados.append({
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'cross_validation_accuracy': cv_score
    })
    
    print('\nModelo:', nombre)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

pd.DataFrame(resultados)

### Comentario de métricas

- **Accuracy** mide el porcentaje total de predicciones correctas.
- **Precision macro** mide qué tan exactas son las predicciones por clase, promediando todas las clases por igual.
- **Recall macro** mide qué tan bien el modelo identifica los casos reales de cada clase.
- **F1 macro** combina precision y recall, siendo útil cuando existen varias clases.

La validación cruzada permite obtener una evaluación más estable del modelo, ya que entrena y evalúa usando diferentes particiones del dataset.

## Guardado de modelos en formato PKL

In [ ]:
os.makedirs('modelos', exist_ok=True)

joblib.dump(modelo_arbol, 'modelos/modelo_arbol_decision.pkl')
joblib.dump(modelo_random_forest, 'modelos/modelo_random_forest.pkl')

print('Modelos guardados correctamente.')